# Module 1: Retail Company Universe and SEC Identifiers

**基础版：** 保留原来的 8 个代码格，使用普通循环，减少重复检查。
说明用中文；代码、代码注释和运行提示用英文。按 **Shift + Enter** 顺序运行。

目标：提出候选公司名单，补上 SEC 编号，保存清单供模块 2 使用。
公司身份匹配成功不等于业务和财务数据通过审核。

你已经跑通旧版，因此**不必为了开始模块 2 重新下载名录**。旧版生成的清单可以直接使用。
此基础版保留原来的 CSV 文件名、主要列名及缓存格式；运行时会覆盖同路径的清单。
代码未在本机连接 SEC 执行，没有预填下载结果。

来源：[SEC API 文档](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)、
[SEC 自动访问说明](https://www.sec.gov/about/developer-resources)。


## Cell 1: Install Packages

安装获取数据和整理表格的工具。已安装时可跳过。


In [ ]:
%pip install requests pandas


## Cell 2: Set Up the Project

导入工具并建立保存目录。`Path.cwd()` 是 Jupyter 的当前运行目录，请检查打印的路径。
若它是 `outputs`，向上一层作为项目目录。也可直接将 `PROJECT_ROOT` 设为你自己的项目路径。
`raw` 存原始来源，`processed` 存整理后的表；`exist_ok=True` 允许目录已存在。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import requests
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "work" / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_ROOT)


## Cell 3: Configure SEC Requests

换成自己的联系邮箱。请求头用于标识程序，不是 API key。
默认读取缓存；需要更新名录时才将 `REFRESH` 改为 `True`。
本格仅检查是否仍使用占位邮箱，不做复杂格式检查。


In [ ]:
SEC_CONTACT = "your_email@example.com"  # Replace with your contact email
REFRESH = False
if SEC_CONTACT == "your_email@example.com":
    raise ValueError("Replace the placeholder with your contact email.")

SEC_HEADERS = {
    "User-Agent": f"RetailInventoryResearch/0.1 (contact: {SEC_CONTACT})",
    "Accept": "application/json",
}
SEC_TICKERS_URL = "https://www.sec.gov/files/company_tickers_exchange.json"


## Cell 4: Define the Candidates

先定义分组和代码，再使用两个普通 `for` 循环，每家公司生成一条记录。
`rows.append(...)` 把记录放进列表，`pd.DataFrame(rows)` 将列表转成表格。
分组是我们的初步研究设定，后续仍需核实。这里仅检查代码不重复，不固定检查必须有 42 家。


In [ ]:
candidate_groups = {
    "Apparel, Footwear, and Off-Price Retail": "TJX ROST BURL LULU GAP URBN ANF BOOT AEO BKE CTRN JILL SFIX SHOE LE CURV ZUMZ DLTH TLYS AKA CATO LVLU DXLG".split(),
    "Department Stores": "M KSS DDS".split(),
    "Home Furnishings, Furniture, and Home Improvement": "HD LOW FND HVT WSM RH ARHS".split(),
    "Sporting Goods, Electronics, and Other Merchandise": "DKS ASO SPWH BBY GME FIVE".split(),
    "Beauty and Personal Care": "ULTA SBH BBWI".split(),
}
rows = []
for subgroup, tickers in candidate_groups.items():
    for ticker in tickers:
        rows.append({"ticker": ticker, "subgroup": subgroup})

candidates = pd.DataFrame(rows)
assert candidates["ticker"].is_unique, "Duplicate stock symbols found."
print("Candidate count:", len(candidates))
display(candidates.head())


## Cell 5: Load the SEC Directory

先读缓存；没有缓存或要求刷新时才下载。只保留 HTTP 状态检查。
`response.json()` 将响应转成 Python 对象；`json.loads` 读取文件中的 JSON 文字。
快照记录来源和获取时间，再用 `fields` 和 `data` 创建官方表。
遇到 403、429 或超时先暂停，不要连续重试；数据结构异常会由 Python/pandas 的原生报错提示。


In [ ]:
cache_path = RAW_DIR / "sec_company_tickers_exchange_snapshot.json"
if cache_path.exists() and not REFRESH:
    snapshot = json.loads(cache_path.read_text(encoding="utf-8"))
    print("Source: local cache")
else:
    response = requests.get(SEC_TICKERS_URL, headers=SEC_HEADERS, timeout=30)
    if response.status_code != 200:
        raise RuntimeError(f"SEC returned HTTP {response.status_code}. Stop and retry later.")
    snapshot = {
        "source_url": SEC_TICKERS_URL,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "payload": response.json(),
    }
    cache_path.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")
    print("Source: fresh SEC download")

payload = snapshot["payload"]
sec_tickers = pd.DataFrame(payload["data"], columns=payload["fields"])
display(sec_tickers.head())


## Cell 6: Match Company Identifiers

先选择官方列并改名，再按 `ticker` 左连接，保留未匹配候选。
`validate="one_to_one"` 防止重复代码使连接产生多行。
`Int64` 允许整数编号缺失；转成字符串后，`.str.zfill(10)` 在左边补零，无需 `lambda`。
`.map` 将连接状态改为项目状态。业务与财务审核状态仍为待审核。


In [ ]:
official = sec_tickers[sec_tickers["ticker"].isin(candidates["ticker"])]
official = official[["ticker", "cik", "name", "exchange"]]
official = official.rename(columns={"name": "sec_company_name"})
universe = candidates.merge(
    official, on="ticker", how="left", indicator=True, validate="one_to_one"
)
universe["cik"] = universe["cik"].astype("Int64")
universe["cik10"] = universe["cik"].astype("string").str.zfill(10)
status_mapping = {"both": "matched", "left_only": "needs_review", "right_only": "needs_review"}
universe["identity_status"] = universe["_merge"].map(status_mapping).astype("string")
universe["eligibility_status"] = "pending_business_and_data_audit"
universe["identity_snapshot_at_utc"] = snapshot["retrieved_at_utc"]
universe = universe.drop(columns="_merge")
display(universe.head())


## Cell 7: Review the Result

这格仅展示数量和待核对公司，不重复检查编号格式。
`matched.sum()` 统计匹配数，`~matched` 筛出未匹配公司。
如果待核对表为空，表示全部候选匹配成功；仍不代表财务数据完整。


In [ ]:
matched = universe["identity_status"] == "matched"
print("Candidate count:", len(universe))
print("Matched count:", int(matched.sum()))
display(universe.loc[~matched, ["ticker", "identity_status"]])


## Cell 8: Save the Universe

保存 CSV 和简短来源说明，不再重复读取比对。下个模块读取时指定 `cik10` 为字符串即可保留前导零。
这些文件写入上面打印的项目目录。公开 Notebook 前记得将联系邮箱恢复成占位值。


In [ ]:
universe_path = PROCESSED_DIR / "retail_candidate_universe.csv"
universe.to_csv(universe_path, index=False, encoding="utf-8-sig")
metadata = {
    "source_url": SEC_TICKERS_URL,
    "source_retrieved_at_utc": snapshot["retrieved_at_utc"],
    "candidate_count": len(universe),
    "business_and_data_eligibility_audited": False,
    "sampling_note": "Current listed-company candidates; historical exits are not yet included.",
    "grouping_note": "Research-defined categories pending annual-report review.",
}
metadata_path = PROCESSED_DIR / "retail_candidate_universe_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved universe:", universe_path)
